<a href="https://colab.research.google.com/github/poggerssLL/Automatica---Grupo-2/blob/main/etapa-02-grafos/13_Algoritmos_de_Busca_BFS_e_DFS_em_Tubulacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 13 - Notebook: Busca em Largura (BFS) e Busca em Profundidade (DFS) na Máquina Envasadora de Copos

Neste notebook implementamos os algoritmos **BFS** e **DFS** para descobrir caminhos de fluxo de material e enumerar rotas de contingência em uma linha de envase de copos.

In [2]:
from collections import deque
from typing import List, Set, Optional

class GrafoEnvasadora:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_transicao(self, origem: str, destino: str, tempo_s: float, tag_valvula: str):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = tempo_s
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Tempo (s)": tempo_s,
            "Válvula": tag_valvula
        })

def criar_rede_envasadora() -> GrafoEnvasadora:
    nos = ["Reservatorio", "Bomba", "V1", "V2", "Envasadora", "Estacao_Inspecao", "Empacotadora"]
    g = GrafoEnvasadora(nos)
    g.adicionar_transicao("Reservatorio", "Bomba", 5, "V-RES-B")
    g.adicionar_transicao("Bomba", "V1", 2, "V-B-1")
    g.adicionar_transicao("Bomba", "V2", 3, "V-B-2")
    g.adicionar_transicao("V1", "Envasadora", 4, "V-1-ENV")
    g.adicionar_transicao("V2", "Envasadora", 6, "V-2-ENV")
    g.adicionar_transicao("Envasadora", "Estacao_Inspecao", 3, "V-ENV-INS")
    g.adicionar_transicao("Estacao_Inspecao", "Empacotadora", 2, "V-INS-EMP")
    return g

class NavegadorGrafos:
    def __init__(self, grafo: GrafoEnvasadora):
        self.g = grafo

    def bfs_menor_numero_etapas(self, origem: str, destino: str, nos_bloqueados: Optional[Set[str]] = None) -> Optional[List[str]]:
        if nos_bloqueados is None:
            nos_bloqueados = set()
        if origem in nos_bloqueados or destino in nos_bloqueados:
            return None
        fila = deque([(origem, [origem])])
        visitados = {origem}
        while fila:
            u_nome, caminho = fila.popleft()
            if u_nome == destino:
                return caminho
            u_idx = self.g.v_to_idx[u_nome]
            for v_idx in range(self.g.n):
                v_nome = self.g.idx_to_v[v_idx]
                if self.g.adj_binaria[u_idx][v_idx] == 1 and v_nome not in visitados and v_nome not in nos_bloqueados:
                    visitados.add(v_nome)
                    fila.append((v_nome, caminho + [v_nome]))
        return None

    def dfs_todos_caminhos(self, origem: str, destino: str, nos_bloqueados: Optional[Set[str]] = None) -> List[List[str]]:
        if nos_bloqueados is None:
            nos_bloqueados = set()
        caminhos: List[List[str]] = []
        def dfs(u_nome: str, visitados: List[str]):
            if u_nome == destino:
                caminhos.append(list(visitados))
                return
            u_idx = self.g.v_to_idx[u_nome]
            for v_idx in range(self.g.n):
                v_nome = self.g.idx_to_v[v_idx]
                if self.g.adj_binaria[u_idx][v_idx] == 1 and v_nome not in visitados and v_nome not in nos_bloqueados:
                    visitados.append(v_nome)
                    dfs(v_nome, visitados)
                    visitados.pop()
        dfs(origem, [origem])
        return caminhos

# Demonstração
rede = criar_rede_envasadora()
nav = NavegadorGrafos(rede)
rota_bfs = nav.bfs_menor_numero_etapas("Reservatorio", "Empacotadora")
print("1. Rota mínima (BFS):", " -> ".join(rota_bfs))
rotas_dfs = nav.dfs_todos_caminhos("Reservatorio", "Empacotadora")
print("2. Total de rotas (DFS):", len(rotas_dfs))
for i, r in enumerate(rotas_dfs, 1):
    print(f"   Opção {i}: {' -> '.join(r)}")

# Simulação de falha na válvula V1
rota_bloqueada = nav.bfs_menor_numero_etapas("Reservatorio", "Empacotadora", nos_bloqueados={"V1"})
print("3. Rota com V1 bloqueada:", " -> ".join(rota_bloqueada) if rota_bloqueada else "Nenhum caminho disponível")


1. Rota mínima (BFS): Reservatorio -> Bomba -> V1 -> Envasadora -> Estacao_Inspecao -> Empacotadora
2. Total de rotas (DFS): 2
   Opção 1: Reservatorio -> Bomba -> V1 -> Envasadora -> Estacao_Inspecao -> Empacotadora
   Opção 2: Reservatorio -> Bomba -> V2 -> Envasadora -> Estacao_Inspecao -> Empacotadora
3. Rota com V1 bloqueada: Reservatorio -> Bomba -> V2 -> Envasadora -> Estacao_Inspecao -> Empacotadora
